# 06 - Forecast Review
**Questions:** how accurate is the forecast, which categories are hard to predict, and what does the next period look like?

Sources: the `forecast.*` tables written by `python -m forecasting.train`, plus `marts.mart_weekly_category_demand`.
WAPE = total absolute error / total actual units (lower is better). Bias = signed error as a share of actuals (negative = under-forecasting).

In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from ingestion.config import get_warehouse_path  # noqa: E402

con = duckdb.connect(str(get_warehouse_path()), read_only=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")


def q(sql: str) -> pd.DataFrame:
    """Run SQL against the warehouse and return a DataFrame."""
    return con.execute(sql).df()

## 1. Model comparison on the holdout
The holdout uses the **latest** backtest origins, which were never used to choose models, so this is the honest test.
`selected` is the per-category policy (a candidate model, or a baseline fallback).

In [ ]:
acc = q("SELECT * FROM forecast.accuracy_summary")
holdout = acc[acc.scope == "holdout_origins"].sort_values("wape")
display(holdout[["model", "rows", "wape", "bias", "mae", "median_category_wape"]])
px.bar(holdout, x="model", y="wape", text="wape", title="Holdout WAPE by model (lower is better)").show()

The seasonal-naive model needs a full year of history, so it is scored on fewer rows. The table below compares every model on **the same origins** (those with at least a year of history), so the comparison is fair.

In [ ]:
display(acc[acc.scope == "full_year_origins"].sort_values("wape")[["model", "rows", "wape", "bias"]])

## 2. Does accuracy get worse further ahead?

In [ ]:
hz = q("SELECT * FROM forecast.horizon_accuracy")
px.line(hz, x="horizon", y="wape", color="model", markers=True,
        title="Holdout WAPE by weeks ahead").show()

## 3. The holiday spike
Every model under-forecasts a demand spike it has no way to anticipate. This shows how badly, using the Black Friday weeks.

In [ ]:
peak = acc[acc.scope == "peak_weeks"].sort_values("wape")
if peak.empty:
    print("No backtest rows fell in the peak period.")
else:
    display(peak[["model", "rows", "wape", "bias"]])
    print("A strongly negative bias means every model forecast far below what actually sold.")

## 4. Actual demand and forecast, by category

In [ ]:
fc = q("SELECT * FROM forecast.weekly_forecast")
hist = q("""
    SELECT week_start, category, units FROM marts.mart_weekly_category_demand
    WHERE category_revenue_rank <= (SELECT categories FROM forecast.run_log)
""")
top = q("SELECT category FROM forecast.category_uncertainty ORDER BY revenue_rank LIMIT 6")["category"].tolist()
for cat in top:
    h = hist[hist.category == cat].sort_values("week_start")
    f = fc[fc.category == cat].sort_values("week_start")
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=h.week_start, y=h.units, name="actual", mode="lines"))
    fig.add_trace(go.Scatter(x=f.week_start, y=f.upper_units, mode="lines", line=dict(width=0),
                             showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=f.week_start, y=f.lower_units, mode="lines", line=dict(width=0),
                             fill="tonexty", name="error band", hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=f.week_start, y=f.forecast_units, name="forecast", mode="lines+markers"))
    fig.update_layout(title=f"{cat}: weekly units and 8-week forecast", height=380)
    fig.show()

## 5. Which categories are hardest to forecast?
Wide bands and high holdout WAPE mean a category's forecast should be used with caution (for example, carry more safety stock).

In [ ]:
unc = q("SELECT * FROM forecast.category_uncertainty ORDER BY uncertainty_rank")
display(unc)
px.bar(unc, x="category", y="holdout_wape", color="chosen_model",
       title="Holdout WAPE by category (higher = harder to forecast)").show()

## 6. Selected policy vs the baselines, per category

In [ ]:
ca = q("SELECT category, model, wape FROM forecast.category_accuracy")
pivot = ca.pivot(index="category", columns="model", values="wape")
pivot["selected_beats_best_baseline"] = pivot["selected"] < pivot[["naive", "moving_average_4"]].min(axis=1)
display(pivot)
print("Categories where the selected policy beat the best baseline on the holdout:",
      int(pivot["selected_beats_best_baseline"].sum()), "of", len(pivot))

## How to read this
- If no model clearly beats the moving-average baseline, say so. A simple baseline that is hard to beat is a legitimate finding.
- A band that is very wide relative to the forecast means demand in that category is too noisy for precise planning.
- Compare the selected policy with the best single model: if a single model does better, per-category choice may be fitting noise.
- Record the headline numbers in `docs/forecast_report.md`.